<a href="https://colab.research.google.com/github/wtraquinas/project-3-brief-NLP-automated-customers-reviews/blob/main/05_Deployment/5_deploymodel_2_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re
import zipfile
zip_path = "original_kaggle_data_download_archive.zip"
with zipfile.ZipFile(zip_path) as z:
    print(z.namelist())  # See what files are inside
    f1 = pd.read_csv(z.open("1429_1.csv"))
    f2 = pd.read_csv(z.open("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv"))
    f3 = pd.read_csv(z.open("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"))

['1429_1.csv', 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv', 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv']


/tmp/ipykernel_10416/1343248299.py:8: DtypeWarning: Columns (1,10) have mixed types. Specify dtype option on import or set low_memory=False.
  f1 = pd.read_csv(z.open("1429_1.csv"))


In [2]:
"""
Data preparation for the Automated Customer Reviews project.
Merges the 3 raw Datafiniti/Amazon review exports, deduplicates, cleans the
corrupted `name` field, maps star ratings to sentiment classes, and writes
a single clean dataset for downstream modeling.


"""
OUT_PATH = "clean_reviews.csv"

print(f"f1 {f1.shape}, f2 {f2.shape}, f3 {f3.shape}")

# ---- Keep only columns relevant to NLP tasks ----
keep_cols = ["id", "name", "asins", "brand", "categories", "primaryCategories",
             "reviews.date", "reviews.rating", "reviews.text", "reviews.title"]

for df in (f1, f2, f3):
    for c in keep_cols:
        if c not in df.columns:
            df[c] = np.nan

merged = pd.concat([f1[keep_cols], f2[keep_cols], f3[keep_cols]], ignore_index=True)
print(f"merged raw {merged.shape}")

# ---- Deduplicate: same product + review text + rating ----
merged = merged.dropna(subset=["reviews.text", "reviews.rating"])
merged = merged.drop_duplicates(subset=["id", "reviews.text", "reviews.rating"])
print(f"after dedup {merged.shape}")
print(f"unique product IDs: {merged['id'].nunique()}")
print(f"unique product names (raw): {merged['name'].nunique()}")

# ---- Fix corrupted `name` field (two names concatenated together) ----
def clean_name(name):
    if pd.isna(name):
        return name
    name = str(name)
    # Datafiniti corruption pattern: "Name A,,,,Name B" or names glued with commas
    # Keep first product-name segment (before a run of 2+ commas, or before an
    # obvious second product name starting after a comma cluster)
    parts = re.split(r",{2,}", name)
    first = parts[0].strip()
    return first

merged["name_clean"] = merged["name"].apply(clean_name)
print(f"unique product names (cleaned): {merged['name_clean'].nunique()}")

# ---- Sentiment mapping ----
def map_sentiment(rating):
    if pd.isna(rating):
        return np.nan
    rating = int(rating)
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

merged["sentiment"] = merged["reviews.rating"].apply(map_sentiment)
merged = merged.dropna(subset=["sentiment"])

# ---- Combine title + text as model input ----
merged["review_text_full"] = (
    merged["reviews.title"].fillna("") + ". " + merged["reviews.text"].fillna("")
).str.strip()
merged = merged[merged["review_text_full"].str.len() > 3]

print("\nFinal dataset:")
print(f"  rows: {len(merged)}")
print(f"  unique product IDs: {merged['id'].nunique()}")
print(f"  unique cleaned product names: {merged['name_clean'].nunique()}")
print("\nSentiment class distribution:")
print(merged["sentiment"].value_counts())
print(merged["sentiment"].value_counts(normalize=True).round(3))

merged.to_csv(OUT_PATH, index=False)
print(f"\nSaved -> {OUT_PATH}")

f1 (34660, 21), f2 (5000, 24), f3 (28332, 24)
merged raw (67992, 10)
after dedup (59743, 10)
unique product IDs: 89
unique product names (raw): 119
unique product names (cleaned): 109

Final dataset:
  rows: 59743
  unique product IDs: 89
  unique cleaned product names: 109

Sentiment class distribution:
sentiment
positive    54841
neutral      2588
negative     2314
Name: count, dtype: int64
sentiment
positive    0.918
neutral     0.043
negative    0.039
Name: proportion, dtype: float64

Saved -> clean_reviews.csv


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("clean_reviews.csv")
df = df.dropna(subset=["review_text_full", "sentiment"])

X = df["review_text_full"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))

vectorizer = TfidfVectorizer(
    max_features=30000, ngram_range=(1, 2), min_df=3, stop_words="english"
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

Train: 47794, Test: 11949
Train class balance:
 sentiment
positive    0.918
neutral     0.043
negative    0.039
Name: proportion, dtype: float64


In [4]:
"""Upload clean_reviews.csv"""

!pip install -q transformers datasets accelerate evaluate scikit-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from datasets import Dataset, ClassLabel
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, DataCollatorWithPadding)
import evaluate

MODEL_NAME = "distilbert-base-uncased"
LABELS = ["negative", "neutral", "positive"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

df = pd.read_csv("clean_reviews.csv").dropna(subset=["review_text_full", "sentiment"])
df["label"] = df["sentiment"].map(label2id)

#Downsample the majority "positive" class to reduce imbalance
# (uncomment if training time/imbalance is an issue)
neg = df[df.sentiment == "negative"]
neu = df[df.sentiment == "neutral"]
pos = df[df.sentiment == "positive"].sample(n=len(neg) * 5, random_state=42)
df = pd.concat([neg, neu, pos]).sample(frac=1, random_state=42)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)

train_ds = Dataset.from_pandas(train_df[["review_text_full", "label"]].rename(
    columns={"review_text_full": "text"}), preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["review_text_full", "label"]].rename(
    columns={"review_text_full": "text"}), preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

Map:   0%|          | 0/13177 [00:00<?, ? examples/s]

Map:   0%|          | 0/3295 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ---- Final evaluation ----
preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
labels_true = preds_output.label_ids

print(classification_report(labels_true, preds, target_names=LABELS, digits=3))
cm = confusion_matrix(labels_true, preds)
print(pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS],
                    columns=[f"pred_{l}" for l in LABELS]))

trainer.save_model("./sentiment_model_final")
tokenizer.save_pretrained("./sentiment_model_final")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.456100,0.296470,0.893778,0.810963
2,0.265100,0.277446,0.906829,0.840903
3,0.206900,0.299821,0.910774,0.851118


              precision    recall  f1-score   support

    negative      0.869     0.877     0.873       463
     neutral      0.753     0.693     0.722       518
    positive      0.951     0.966     0.959      2314

    accuracy                          0.911      3295
   macro avg      0.858     0.845     0.851      3295
weighted avg      0.908     0.911     0.909      3295

               pred_negative  pred_neutral  pred_positive
true_negative            406            46             11
true_neutral              55           359            104
true_positive              6            72           2236


('./sentiment_model_final/tokenizer_config.json',
 './sentiment_model_final/special_tokens_map.json',
 './sentiment_model_final/vocab.txt',
 './sentiment_model_final/added_tokens.json',
 './sentiment_model_final/tokenizer.json')

## Export for deployment (ONNX + quantization)

Convert the fine-tuned checkpoint to ONNX and apply dynamic INT8 quantization
so the model is small and fast enough to run in a serverless function (e.g.
Vercel). This shrinks the model from ~260 MB (PyTorch fp32) to ~65 MB and
drops the runtime dependency on PyTorch (inference only needs `onnxruntime`).

In [6]:
import os


from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from transformers import AutoTokenizer

SRC_DIR = "./sentiment_model_final"
ONNX_DIR = "./sentiment_model_onnx"

# ---- 1. Export the fine-tuned checkpoint to ONNX ----
# Check if SRC_DIR exists, assuming previous cells have created it.
if not os.path.exists(SRC_DIR):
    print(f"Error: Source directory '{SRC_DIR}' not found. Please ensure the previous model training and saving steps have completed successfully.")
else:
    ort_model = ORTModelForSequenceClassification.from_pretrained(SRC_DIR, export=True)
    tokenizer = AutoTokenizer.from_pretrained(SRC_DIR)

    os.makedirs(ONNX_DIR, exist_ok=True) # Ensure the ONNX output directory exists
    ort_model.save_pretrained(ONNX_DIR)
    tokenizer.save_pretrained(ONNX_DIR)

    # ---- 2. Dynamic INT8 quantization (CPU-friendly, no calibration data needed) ----
    quantizer = ORTQuantizer.from_pretrained(ONNX_DIR, file_name="model.onnx")
    qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=ONNX_DIR, quantization_config=qconfig)

    print("Exported files:")
    for fname in sorted(os.listdir(ONNX_DIR)):
        size_mb = os.path.getsize(os.path.join(ONNX_DIR, fname)) / 1e6
        print(f"  {fname}: {size_mb:.1f} MB")

    print(f"\nCopy the contents of {ONNX_DIR}/ into the Vercel project's model/ folder.")

Multiple distributions found for package optimum. Picked distribution: optimum-onnx
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Exported files:
  config.json: 0.0 MB
  model.onnx: 267.9 MB
  model_quantized.onnx: 67.4 MB
  ort_config.json: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 0.7 MB
  tokenizer_config.json: 0.0 MB
  vocab.txt: 0.2 MB

Copy the contents of ./sentiment_model_onnx/ into the Vercel project's model/ folder.
